In [0]:
import io
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--target", "/tmp/xllib",
                "openpyxl==3.1.5"], check=True)
if "/tmp/xllib" not in sys.path:
    sys.path.insert(0, "/tmp/xllib")

import requests
import pandas as pd

url = "https://www.eia.gov/outlooks/steo/archives/jan25_base.xlsx"
r = requests.get(url, headers={"User-Agent": "oil-shock-portfolio/1.0"}, timeout=60)
print("HTTP", r.status_code, "| size:", round(len(r.content) / 1e6, 2), "MB")

xl = pd.ExcelFile(io.BytesIO(r.content), engine="openpyxl")
print("Sheets:", xl.sheet_names)

found = None
for sh in xl.sheet_names:
    raw = xl.parse(sh, header=None)
    hits = raw.index[raw.iloc[:, 0].astype(str).str.strip().isin(["BREPUUS", "WTIPUUS"])].tolist()
    if hits:
        found = (sh, raw, hits)
        break

if found:
    sh, raw, hits = found
    print(f"\nFound in sheet '{sh}', rows {hits}")
    print("\nFirst 6 rows, first 8 columns:")
    display(raw.iloc[:6, :8])
    print("\nBrent / WTI rows, first 8 columns:")
    display(raw.iloc[hits, :8])
else:
    print("BREPUUS not found in column A of any sheet")

display(spark.sql("""
    SELECT series_id, series_label, COUNT(*) AS rows,
           MIN(release_month) AS first_release, MAX(release_month) AS last_release
    FROM oil_shock.gold.fct_steo_forecast
    WHERE series_id LIKE 'BRE%' OR series_id LIKE 'WTI%'
    GROUP BY series_id, series_label
"""))

In [0]:
import time
from datetime import datetime, timezone

MON_ABBR = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
MON_NUM = {m.capitalize(): i + 1 for i, m in enumerate(MON_ABBR)}
SERIES = {"BREPUUS", "WTIPUUS"}

def parse_steo(content, release):
    xl = pd.ExcelFile(io.BytesIO(content), engine="openpyxl")
    order = ([s for s in ["1tab", "2tab"] if s in xl.sheet_names] +
             [s for s in xl.sheet_names if s not in ("1tab", "2tab", "Dates", "Contents")])
    need, rows = set(SERIES), []
    for sh in order:
        if not need:
            break
        raw = xl.parse(sh, header=None)
        ids = raw.iloc[:, 0].astype(str).str.strip()
        hits = [(i, ids[i]) for i in raw.index if ids[i] in need]
        if not hits:
            continue
        month_row = next((i for i in range(min(12, len(raw)))
                          if (raw.iloc[i, 2:].astype(str).str.strip() == "Jan").sum() >= 2), None)
        if month_row is None:
            continue
        years = pd.to_numeric(raw.iloc[month_row - 1], errors="coerce").ffill()
        months = raw.iloc[month_row].astype(str).str.strip()
        cols = [j for j in range(2, raw.shape[1]) if months[j] in MON_NUM and pd.notna(years[j])]
        for i, sid in hits:
            if sid not in need:
                continue
            for j in cols:
                v = pd.to_numeric(raw.iat[i, j], errors="coerce")
                if pd.notna(v):
                    rows.append((release, sid, f"{int(years[j])}-{MON_NUM[months[j]]:02d}", float(v)))
            need.discard(sid)
    return rows, need

releases = [f"{y}-{m:02d}" for y in range(2015, 2027) for m in range(1, 13) if f"{y}-{m:02d}" <= "2026-09"]
all_rows, log = [], []
t0 = time.time()
for rel in releases:
    y, m = rel.split("-")
    fname = f"{MON_ABBR[int(m) - 1]}{y[2:]}_base.xlsx"
    url = f"https://www.eia.gov/outlooks/steo/archives/{fname}"
    try:
        resp = requests.get(url, headers={"User-Agent": "oil-shock-portfolio/1.0"}, timeout=60)
        if resp.status_code != 200:
            log.append((rel, fname, f"HTTP {resp.status_code}", 0))
            continue
        rows, missing = parse_steo(resp.content, rel)
        all_rows += [r + (fname,) for r in rows]
        log.append((rel, fname, "ok" if not missing else f"missing {sorted(missing)}", len(rows)))
    except Exception as e:
        log.append((rel, fname, f"error: {str(e)[:80]}", 0))
    time.sleep(0.5)

print(f"Done in {time.time() - t0:.0f}s")
logdf = pd.DataFrame(log, columns=["release_month", "file", "status", "rows"])
print("Files ok:", (logdf["status"] == "ok").sum(), "of", len(logdf))
failed = logdf[logdf["status"] != "ok"]
if len(failed):
    display(failed)
else:
    print("No failed files.")

arch = pd.DataFrame(all_rows, columns=["release_month", "series_id", "target_month", "value", "source_file"])
arch["loaded_at_utc"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
(spark.createDataFrame(arch)
     .write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable("oil_shock.bronze.steo_archive"))
print("Saved oil_shock.bronze.steo_archive rows:", len(arch))

# Check: what each 2026 release expected Brent to average in Dec 2026
display(spark.sql("""
    SELECT release_month, ROUND(value, 2) AS brent_forecast_for_dec_2026
    FROM oil_shock.bronze.steo_archive
    WHERE series_id = 'BREPUUS' AND target_month = '2026-12' AND release_month >= '2025-10'
    ORDER BY release_month
"""))

In [0]:
print("Rows parsed:", len(all_rows), "| files:", len(log))

arch = pd.DataFrame(all_rows, columns=["release_month", "series_id", "target_month", "value", "source_file"])
arch["loaded_at_utc"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
(spark.createDataFrame(arch)
     .write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable("oil_shock.bronze.steo_archive"))
print("Saved oil_shock.bronze.steo_archive rows:", len(arch))

print(arch.groupby("series_id").agg(releases=("release_month", "nunique"),
                                    first_release=("release_month", "min"),
                                    last_release=("release_month", "max")))

# Check: what each release expected Brent to average in Dec 2026
display(spark.sql("""
    SELECT release_month, ROUND(value, 2) AS brent_forecast_for_dec_2026
    FROM oil_shock.bronze.steo_archive
    WHERE series_id = 'BREPUUS' AND target_month = '2026-12' AND release_month >= '2025-10'
    ORDER BY release_month
"""))